# AI TDD Orchestrator - Free GPU Server

This notebook turns a **free Google Colab T4 GPU** into an Ollama API server.
Once running, your GitHub Actions pipeline automatically uses this GPU.

### Steps:
1. Make sure **T4 GPU** is selected (Runtime > Change runtime type > T4 GPU)
2. Run all cells (Runtime > Run all)
3. Copy the ngrok URL printed in Cell 4
4. Add it as `COLAB_OLLAMA_URL` secret in your GitHub repo
5. Done! Your pipeline now uses a free T4 GPU!

## Cell 1: Install Dependencies and Ollama

In [ ]:
# Install zstd (required by Ollama) and pciutils (GPU detection)
!sudo apt-get update -qq && sudo apt-get install -y -qq zstd pciutils > /dev/null 2>&1
print('[OK] Dependencies installed')

# Verify GPU is visible
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || echo 'WARNING: No GPU detected. Select Runtime > Change runtime type > T4 GPU'

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh
print('[OK] Ollama installed')

## Cell 2: Start Ollama Server and Pull Model

In [ ]:
import subprocess, time, os

# Start Ollama in background using full path
env = os.environ.copy()
env['OLLAMA_HOST'] = '0.0.0.0'
process = subprocess.Popen(
    ['/usr/local/bin/ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    env=env,
)
time.sleep(5)
print('[OK] Ollama server started')

# Pull the coding model (7b fits in T4 15GB VRAM)
!ollama pull qwen2.5-coder:7b

# Verify models loaded
import requests
try:
    tags = requests.get('http://localhost:11434/api/tags').json()
    names = [m['name'] for m in tags.get('models', [])]
    print(f'[OK] Models loaded: {names}')
except Exception as e:
    print(f'[ERROR] Could not verify models: {e}')

## Cell 3: Install ngrok

Get your **free** ngrok auth token at: https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
!pip install pyngrok -q
print('[OK] ngrok installed')

## Cell 4: Create Public Tunnel

Paste your ngrok auth token below, then run this cell.

**Copy the printed URL and add it as a GitHub secret called `COLAB_OLLAMA_URL`**

In [ ]:
from pyngrok import ngrok

# ==================================================
# PASTE YOUR FREE NGROK AUTH TOKEN HERE:
NGROK_TOKEN = ''  # <-- Get from https://dashboard.ngrok.com
# ==================================================

if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)
else:
    print('[WARNING] No ngrok token set.')
    print('Get one free at https://dashboard.ngrok.com')
    print('Paste it in NGROK_TOKEN above, then re-run this cell.')
    print()

# Create tunnel
tunnel = ngrok.connect(11434)
public_url = tunnel.public_url

print()
print('=' * 60)
print('  YOUR FREE GPU OLLAMA SERVER IS LIVE!')
print('=' * 60)
print()
print(f'  Public URL: {public_url}')
print()
print(f'  Add this as GitHub Secret:')
print(f'    Secret Name:  COLAB_OLLAMA_URL')
print(f'    Secret Value: {public_url}')
print()
print(f'  Or set directly in workflow env:')
print(f'    OLLAMA_URL: {public_url}/api/generate')
print()
print(f'  Server stays alive while this Colab tab is open.')
print(f'  Free tier: ~12 hours max per session')
print('=' * 60)

## Cell 5: Keep Alive

Run this cell to prevent Colab from timing out.
Press **Stop** when done.

In [ ]:
import time, requests

print(f'GPU Server running at: {public_url}')
print('Press Stop to shut down.')
print()

counter = 0
while True:
    counter += 1
    try:
        requests.get('http://localhost:11434/api/tags', timeout=5)
    except Exception:
        pass
    time.sleep(60)
    if counter % 5 == 0:
        print(f'Still alive ({counter} min) | URL: {public_url}')